# 13. Spot and futures on the same events

The spot results have 83 events and the futures results 84, so the two event catalogs differ. In this descriptive notebook I pair the two catalogs by crash (4% breach) time and ask, for the same event, which market's price moves first and which market's V+F signal appears first. My aim is to separate information actually transmitted between markets from simple differences in volatility and thresholds. I use the Binance BTCUSDT spot and futures one-minute caches and the 2023-01-01 to 2026-04-30 evaluation period.

I build or optimize no new policy. I reproduce the existing notebook 05 models, thresholds and alerts (spot 800/39/22, futures 800/35/19) exactly and compare them descriptively only, reporting counts, distributions and orderings instead of tests. I write the results to `btc_lead_cost_results_77months_spot/spot_futures_same_event_v1/`.

The notebook has five parts.
1. Reproduction of the notebook 05 results for both markets (thresholds, alert times, hit counts).
2. Event pairing: breach-time difference ≤60 minutes (sensitivity 15 and 240 minutes), with a reason for each event found in only one market (other market in its 24-hour refractory period / other market also fell 3% or more but did not meet the 4% rule / a different move / data gap).
3. Price ordering in paired events: differences in peak, onset (1%), breach (4%) and confirmation times.
4. Signal ordering in paired events: in a common window (240 to 30 minutes before the earlier breach), each market's first raw signal (score ≥ its own threshold) and first actual alert (with the budget applied), and the first crossing of each component (volatility z_vol, taker sell ratio tsr).
5. Market-wide minute-level lead-lag: lagged correlations of one-minute returns and absolute returns, in pre-event windows and over the whole period.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df


OUT = SPOT_BASE / 'spot_futures_same_event_v1'; OUT.mkdir(parents=True, exist_ok=True)
FUT_CAS = FUT_BASE / 'causal_alert_selection_v1'
FUT_CAT = FUT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv'
SPOT_CAS = SPOT_BASE / 'causal_alert_selection_v1'
for _p in [FUT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet', FUT_BASE / 'cache' / 'btc_signals_1m.parquet', FUT_CAT,
           FUT_CAS / 'alarms_first_eligible.csv', FUT_CAS / 'selection_policy_comparison.csv', FUT_CAS / 'selected_rules_from_2022.csv',
           SPOT_CAS / 'alarms_first_eligible.csv']:
    if not _p.exists():
        raise FileNotFoundError(f'Required file not found: {_p}')
TOL = 60; TOLS = [15, 60, 240]; NEAR_MISS = 0.03; LAGS = list(range(-5, 6))
VENUES = ['spot', 'futures']
SPEC = {'title': 'Spot vs futures on the same crash events (descriptive only)',
        'models': 'notebook-5 V+F per venue, 2021 train, 2022 calibration, calendar 20/month 240/yr; reproduced exactly (guards), no new policy',
        'matching': f'one-to-one greedy on |breach_spot - breach_futures| <= {TOL} min; sensitivity {TOLS}',
        'unmatched_reasons': ['other venue refractory (its last event confirmed within 24 h before)',
                              f'near-miss (other venue drawdown from its 240-min peak >= {NEAR_MISS:.0%} within +-{TOL} min)',
                              'other venue touched 4% but the 5-minute/other rule failed', 'different move', 'other venue data gap'],
        'signal_window': 'common window [min breach - 240, min breach - 30]; onset reference = earlier of the two 1% onsets',
        'components': 'first minute z_vol / tsr >= own 2022 99th percentile in the common window',
        'lead_lag': f'correlation of 1-min returns and absolute returns, futures at t vs spot at t+k, k in {LAGS}; whole evaluation period and pre-event windows',
        'status': 'descriptive; no hypothesis tests'}
(OUT / 'SAME_EVENT_SPEC.json').write_text(json.dumps(SPEC, indent=2, sort_keys=True) + '\n')
print('Output:', OUT)

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/spot_futures_same_event_v1


## 1. Reproduce the notebook 05 results for both markets

In [2]:
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
px, sg = load_btc_spot()
fpx = pd.read_parquet(FUT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet', columns=['close', 'volume']).sort_index()
fsg = pd.read_parquet(FUT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
for fr in (fpx, fsg):
    fr.index = fr.index.tz_convert('UTC')
assert fpx.index.equals(fsg.index)
common = px.index.intersection(fpx.index)
print(f'spot minutes {len(px):,}, futures minutes {len(fpx):,}, common {len(common):,}')

V = {}
for name, P_, G_, cat in [('spot', px, sg, SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv'),
                          ('futures', fpx, fsg, FUT_CAT)]:
    T = P_.index.asi8; close = P_.close.to_numpy(float); volume = P_.volume.to_numpy(float)
    v = G_.z_vol.to_numpy(float); f = G_.tsr.to_numpy(float); h = G_.z_hs.to_numpy(float)
    valid = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v) & np.isfinite(f) & np.isfinite(h) & (f >= 0) & (f <= 1))
    E = load_catalog(cat); y = label_from_catalog(T, E); tr = train_mask(T, valid)
    m = new_logit().fit(np.column_stack([v, f])[tr], y[tr])
    S = np.full(len(T), np.nan); S[valid] = m.predict_proba(np.column_stack([v, f])[valid])[:, 1]
    cal = valid & in_period(T, '2022-01-01', '2023-01-01'); ev = valid & in_period(T, EVAL_START, EVAL_END)
    th, _, _ = calibrate_calendar(T[cal], S[cal])
    alerts = emit_calendar_fast(T[ev], S[ev] >= th)
    V[name] = dict(T=T, close=close, S=S, v=v, f=f, valid=valid, th=th, alerts=alerts, E=E, EV=events_between(E, EVAL_START, EVAL_END),
                   q_v=float(np.nanquantile(v[cal], .99)), q_f=float(np.nanquantile(f[cal], .99)))
    del y, m

sp, fu = V['spot'], V['futures']
st_s = statistics(sp['EV'], sp['alerts'])
saved_s = np.sort(pd.to_datetime(pd.read_csv(SPOT_CAS / 'alarms_first_eligible.csv').alarm_utc, utc=True).astype('int64').to_numpy())
if abs(sp['th'] - TH_CAL) > 1e-12 or not np.array_equal(saved_s, sp['alerts']) or [st_s['alarms'], st_s['caught'], st_s['before_1pct']] != REF['first_eligible']:
    raise RuntimeError('GUARD FAIL: spot notebook-5 result not reproduced')
fsel = pd.read_csv(FUT_CAS / 'selected_rules_from_2022.csv'); fres = pd.read_csv(FUT_CAS / 'selection_policy_comparison.csv')
th_ref = float(fsel.loc[fsel.policy == 'first_eligible', 'param'].iloc[0]); r5 = fres[fres.policy == 'first_eligible'].iloc[0]
st_f = statistics(fu['EV'], fu['alerts'])
saved_f = np.sort(pd.to_datetime(pd.read_csv(FUT_CAS / 'alarms_first_eligible.csv').alarm_utc, utc=True).astype('int64').to_numpy())
if abs(fu['th'] - th_ref) > 1e-12 or not np.array_equal(saved_f, fu['alerts']) or \
        [st_f['alarms'], st_f['caught'], st_f['before_1pct']] != [int(r5.alarms), int(r5.caught), int(r5.before_1pct)] or \
        len(fu['EV']) != int(round(r5.caught / r5.recall)):
    raise RuntimeError(f'GUARD FAIL: futures notebook-5 result not reproduced ({fu["th"]} vs {th_ref}; {st_f})')
REPRO = pd.DataFrame([dict(venue='spot', threshold_2022=sp['th'], events=len(sp['EV']), alarms=st_s['alarms'], caught=st_s['caught'], pre_onset=st_s['before_1pct']),
                      dict(venue='futures', threshold_2022=fu['th'], events=len(fu['EV']), alarms=st_f['alarms'], caught=st_f['caught'], pre_onset=st_f['before_1pct'])])
REPRO['post_onset'] = REPRO.caught - REPRO.pre_onset
REPRO.to_csv(OUT / '1_reproduced_baselines.csv', index=False)
print('PASS: both venues reproduce notebook 5 exactly'); display(REPRO)

spot minutes 3,329,280, futures minutes 3,329,280, common 3,329,280
PASS: both venues reproduce notebook 5 exactly


,venue,threshold_2022,events,alarms,caught,pre_onset,post_onset
0,spot,0.669899,83,800,39,22,17
1,futures,0.671087,84,800,35,19,16


## 2. Event pairing and reasons for unpaired events

In [3]:
def pair_events(Ea, Eb, tol):
    ba = Ea.breach_utc.astype('int64').to_numpy(); bb = Eb.breach_utc.astype('int64').to_numpy()
    cand = sorted((abs(int(x) - int(y)), i, j) for i, x in enumerate(ba) for j, y in enumerate(bb) if abs(int(x) - int(y)) <= tol * NS)
    ua, ub, pairs = set(), set(), []
    for _, i, j in cand:
        if i not in ua and j not in ub:
            ua.add(i); ub.add(j); pairs.append((i, j))
    return sorted(pairs), sorted(set(range(len(Ea))) - ua), sorted(set(range(len(Eb))) - ub)


sens = []
for tol in TOLS:
    pr, oa, ob = pair_events(sp['EV'], fu['EV'], tol)
    sens.append(dict(tolerance_min=tol, matched=len(pr), spot_only=len(oa), futures_only=len(ob)))
SENS = pd.DataFrame(sens); SENS.to_csv(OUT / '2_matching_sensitivity.csv', index=False); display(SENS)
PAIRS, ONLY_S, ONLY_F = pair_events(sp['EV'], fu['EV'], TOL)


def drawdown_from_peak(close, i0, i1):
    """max over t in [i0, i1] of 1 - close_t / max(close over the previous 240 minutes)."""
    lo = max(i0 - LEAD_MAX, 0)
    seg = pd.Series(close[lo:i1 + 1])
    prior = seg.rolling(LEAD_MAX, min_periods=LEAD_MAX // 2).max().shift(1).to_numpy()
    dd = 1 - seg.to_numpy() / prior
    dd = dd[(i0 - lo):]
    return float(np.nanmax(dd)) if np.isfinite(dd).any() else np.nan


def why_unmatched(e, other):
    o = V[other]; b = int(e.breach_utc.value)
    i0 = int(np.searchsorted(o['T'], b - TOL * NS)); i1 = int(np.searchsorted(o['T'], b + TOL * NS))
    gap = int((~np.isfinite(o['close'][max(i0 - LEAD_MAX, 0):i1 + 1])).sum())
    conf = o['E'].confirmed_utc.astype('int64').to_numpy()
    refr = bool(((conf < b - TOL * NS) & (conf >= b - 1440 * NS)).any())
    dd = drawdown_from_peak(o['close'], i0, i1)
    if gap > 5:
        reason = 'other venue data gap'
    elif refr:
        reason = 'other venue refractory (24h)'
    elif np.isfinite(dd) and dd >= 0.04:
        reason = 'other venue touched 4% (5-min/other rule failed)'
    elif np.isfinite(dd) and dd >= NEAR_MISS:
        reason = f'near-miss (other venue {NEAR_MISS:.0%}-4%)'
    else:
        reason = 'different move'
    return dict(missing_minutes_other=gap, other_refractory=refr, other_max_drawdown_pct=dd * 100 if np.isfinite(dd) else np.nan, reason=reason)


un = []
for venue, other, ids in [('spot', 'futures', ONLY_S), ('futures', 'spot', ONLY_F)]:
    for i in ids:
        e = V[venue]['EV'].iloc[i]
        un.append(dict(venue=venue, event_id=int(e.event_id), breach_utc=e.breach_utc, own_decline_pct=float(e.decline_at_breach_pct),
                       **why_unmatched(e, other)))
UN = pd.DataFrame(un); UN.to_csv(OUT / '2_unmatched_events_reasons.csv', index=False)
print(f'tolerance {TOL} min: matched {len(PAIRS)}, spot-only {len(ONLY_S)}, futures-only {len(ONLY_F)}')
display(UN.groupby(['venue', 'reason']).size().rename('events').reset_index())

,tolerance_min,matched,spot_only,futures_only
0,15,82,1,2
1,60,83,0,1
2,240,83,0,1


tolerance 60 min: matched 83, spot-only 0, futures-only 1


,venue,reason,events
0,futures,other venue touched 4% (5-min/other rule failed),1


## 3. Price ordering in paired events (futures − spot, minutes; negative means futures first)

In [4]:
rows = []
for i, j in PAIRS:
    es, ef = sp['EV'].iloc[i], fu['EV'].iloc[j]
    rows.append(dict(spot_event_id=int(es.event_id), futures_event_id=int(ef.event_id), spot_breach_utc=es.breach_utc,
                     **{f'd_{c.replace("_utc", "")}_min': (ef[c] - es[c]).total_seconds() / 60 for c in TCOLS},
                     spot_decline_pct=float(es.decline_at_breach_pct), futures_decline_pct=float(ef.decline_at_breach_pct),
                     spot_onset_to_breach_min=float(es.onset_to_breach_minutes), futures_onset_to_breach_min=float(ef.onset_to_breach_minutes)))
PM = pd.DataFrame(rows); PM.to_csv(OUT / '3_matched_price_timing.csv', index=False)
order_rows = []
for c in TCOLS:
    d = PM[f'd_{c.replace("_utc", "")}_min']
    order_rows.append(dict(milestone=c.replace('_utc', ''), pairs=len(d), futures_first=int((d < 0).sum()), same_minute=int((d == 0).sum()),
                           spot_first=int((d > 0).sum()), median_diff_min=float(d.median()), mean_abs_diff_min=float(d.abs().mean())))
PO = pd.DataFrame(order_rows); PO.to_csv(OUT / '3_matched_price_order_summary.csv', index=False); display(PO)

,milestone,pairs,futures_first,same_minute,spot_first,median_diff_min,mean_abs_diff_min
0,peak,83,7,74,2,0.0,0.975904
1,onset_1pct,83,10,65,8,0.0,1.807229
2,breach,83,6,71,6,0.0,0.795181
3,confirmed,83,6,71,6,0.0,0.795181


## 4. Signal ordering in paired events

In [5]:
def first_in(times, lo, hi):
    j = np.searchsorted(times, lo, 'left')
    return int(times[j]) if j < len(times) and times[j] <= hi else None


def first_cross(vn, arr_key, thr, lo, hi):
    o = V[vn]; i0 = int(np.searchsorted(o['T'], lo, 'left')); i1 = int(np.searchsorted(o['T'], hi, 'right'))
    x = o[arr_key][i0:i1]; ok = o['valid'][i0:i1] & np.isfinite(x) & (x >= thr)
    k = np.flatnonzero(ok)
    return int(o['T'][i0 + k[0]]) if len(k) else None


def who_first(ts_, tf_):
    if ts_ is None and tf_ is None:
        return 'neither'
    if ts_ is None:
        return 'futures only'
    if tf_ is None:
        return 'spot only'
    return 'same minute' if ts_ == tf_ else ('spot first' if ts_ < tf_ else 'futures first')


rows = []
for i, j in PAIRS:
    es, ef = sp['EV'].iloc[i], fu['EV'].iloc[j]
    b0 = min(es.breach_utc.value, ef.breach_utc.value); o0 = min(es.onset_1pct_utc.value, ef.onset_1pct_utc.value)
    lo, hi = b0 - LEAD_MAX * NS, b0 - LEAD_MIN * NS
    rec = dict(spot_event_id=int(es.event_id), futures_event_id=int(ef.event_id), common_breach_utc=pd.Timestamp(b0, tz='UTC'),
               common_onset_utc=pd.Timestamp(o0, tz='UTC'), onset_in_window=bool(lo <= o0 <= hi))
    for kind, getter in [('raw_signal', lambda vn: first_cross(vn, 'S', V[vn]['th'], lo, hi)),
                         ('alert', lambda vn: first_in(V[vn]['alerts'], lo, hi)),
                         ('zvol_q99', lambda vn: first_cross(vn, 'v', V[vn]['q_v'], lo, hi)),
                         ('tsr_q99', lambda vn: first_cross(vn, 'f', V[vn]['q_f'], lo, hi))]:
        t_s, t_f = getter('spot'), getter('futures')
        rec[f'{kind}_order'] = who_first(t_s, t_f)
        for vn, t_ in [('spot', t_s), ('futures', t_f)]:
            rec[f'{kind}_{vn}_min_before_common_onset'] = (o0 - t_) / NS if t_ is not None else np.nan
        rec[f'{kind}_futures_minus_spot_min'] = (t_f - t_s) / NS if (t_s is not None and t_f is not None) else np.nan
    rows.append(rec)
SIG = pd.DataFrame(rows); SIG.to_csv(OUT / '4_matched_signal_timing.csv', index=False)
cats = ['spot first', 'futures first', 'same minute', 'spot only', 'futures only', 'neither']
SO = pd.DataFrame({k: SIG[f'{k}_order'].value_counts().reindex(cats, fill_value=0) for k in ['raw_signal', 'alert', 'zvol_q99', 'tsr_q99']}).T
SO.to_csv(OUT / '4_signal_order_summary.csv'); display(SO)
pre_rows = []
for k in ['raw_signal', 'alert']:
    for vn in VENUES:
        x = SIG[f'{k}_{vn}_min_before_common_onset']
        pre_rows.append(dict(signal=k, venue=vn, events_with_signal=int(x.notna().sum()), before_common_onset=int((x > 0).sum()),
                             after_common_onset=int((x <= 0).sum()), median_min_before_onset=float(x.median()) if x.notna().any() else np.nan))
PRE = pd.DataFrame(pre_rows); PRE.to_csv(OUT / '4_signal_relative_to_onset.csv', index=False); display(PRE)

ok = sp['valid'] & np.isin(sp['T'], fu['T'][fu['valid']]) & in_period(sp['T'], EVAL_START, EVAL_END)
Ss = pd.Series(sp['S'], index=sp['T']); Sf = pd.Series(fu['S'], index=fu['T'])
tt = sp['T'][ok]; a_s = Ss.loc[tt].to_numpy() >= sp['th']; a_f = Sf.loc[tt].to_numpy() >= fu['th']
CO = pd.DataFrame([dict(minutes=len(tt), score_correlation=float(np.corrcoef(Ss.loc[tt], Sf.loc[tt])[0, 1]),
                        both_above=int((a_s & a_f).sum()), spot_only_above=int((a_s & ~a_f).sum()), futures_only_above=int((~a_s & a_f).sum()),
                        jaccard=float((a_s & a_f).sum() / max((a_s | a_f).sum(), 1)))])
CO.to_csv(OUT / '4_score_cofiring.csv', index=False); display(CO)

,spot first,futures first,same minute,spot only,futures only,neither
raw_signal,6,13,49,0,1,14
alert,7,6,22,4,0,44
zvol_q99,1,6,13,0,0,63
tsr_q99,28,7,5,24,5,14


,signal,venue,events_with_signal,before_common_onset,after_common_onset,median_min_before_onset
0,raw_signal,spot,68,39,29,5.0
1,raw_signal,futures,69,39,30,6.0
2,alert,spot,39,22,17,4.0
3,alert,futures,35,19,16,6.0


,minutes,score_correlation,both_above,spot_only_above,futures_only_above,jaccard
0,1750613,0.993131,89877,3180,6579,0.902053


## 5. Minute-level lead-lag (futures at t vs spot at t+k; a larger correlation at k>0 means futures move first)

In [6]:
cidx = sp['T'][np.isin(sp['T'], fu['T'])]
cs = pd.Series(sp['close'], index=sp['T']).loc[cidx]; cf = pd.Series(fu['close'], index=fu['T']).loc[cidx]
rs = np.log(cs).diff().to_numpy(); rf = np.log(cf).diff().to_numpy()
tmask_all = in_period(cidx, EVAL_START, EVAL_END)
win = np.zeros(len(cidx), bool)
for i, j in PAIRS:
    b0 = min(sp['EV'].iloc[i].breach_utc.value, fu['EV'].iloc[j].breach_utc.value)
    win |= (cidx >= b0 - LEAD_MAX * NS) & (cidx <= b0)


def lagcorr(x, y, mask, k):
    # corr(futures_t, spot_{t+k}) over minutes where both are finite and t (and t+k) are inside the mask
    n = len(x)
    if k >= 0:
        a, b_, m = x[:n - k], y[k:], mask[:n - k] & mask[k:]
    else:
        a, b_, m = x[-k:], y[:n + k], mask[-k:] & mask[:n + k]
    g = m & np.isfinite(a) & np.isfinite(b_)
    return float(np.corrcoef(a[g], b_[g])[0, 1]) if g.sum() > 10 else np.nan


rows = []
for label, mask in [('evaluation period', tmask_all), ('pre-event windows (240 min before matched breaches)', win & tmask_all)]:
    for k in LAGS:
        rows.append(dict(sample=label, lag_k=k, corr_returns=lagcorr(rf, rs, mask, k), corr_abs_returns=lagcorr(np.abs(rf), np.abs(rs), mask, k)))
LL = pd.DataFrame(rows); LL.to_csv(OUT / '5_lead_lag_correlations.csv', index=False)
display(LL.pivot(index='lag_k', columns='sample', values=['corr_returns', 'corr_abs_returns']).round(4))

basis = 1e4 * (np.log(cf.to_numpy()) - np.log(cs.to_numpy()))
brow = []
for i, j in PAIRS:
    es, ef = sp['EV'].iloc[i], fu['EV'].iloc[j]
    b0 = min(es.breach_utc.value, ef.breach_utc.value); o0 = min(es.onset_1pct_utc.value, ef.onset_1pct_utc.value)
    get = lambda t_: float(basis[min(np.searchsorted(cidx, t_), len(cidx) - 1)])
    brow.append(dict(spot_event_id=int(es.event_id), basis_bp_at_window_start=get(b0 - LEAD_MAX * NS), basis_bp_at_onset=get(o0), basis_bp_at_breach=get(b0)))
BA = pd.DataFrame(brow); BA['change_window_to_breach_bp'] = BA.basis_bp_at_breach - BA.basis_bp_at_window_start
BA.to_csv(OUT / '5_basis_around_events.csv', index=False); display(BA.describe().round(2))

corr_returns                                                      \
sample evaluation period pre-event windows (240 min before matched breaches)   
lag_k                                                                          
-5               -0.0026                                            -0.0009    
-4               -0.0045                                            -0.0002    
-3               -0.0072                                            -0.0151    
-2               -0.0089                                             0.0221    
-1               -0.0162                                             0.0239    
 0                0.9903                                             0.9905    
 1                0.0055                                             0.0400    
 2               -0.0041                                             0.0351    
 3               -0.0047                                            -0.0046    
 4               -0.0032                                            -0.0059    
 5               -0.0034                                            -0.0091    

        corr_abs_returns                                                      
sample evaluation period pre-event windows (240 min before matched breaches)  
lag_k                                                                         
-5                0.3102                                             0.2668   
-4                0.3148                                             0.2799   
-3                0.3251                                             0.3051   
-2                0.3444                                             0.3206   
-1                0.3838                                             0.3863   
 0                0.9855                                             0.9863   
 1                0.3830                                             0.3948   
 2                0.3449                                             0.3180   
 3                0.3255                                             0.3042   
 4                0.3165                                             0.2953   
 5                0.3124                                             0.2742

,spot_event_id,basis_bp_at_window_start,basis_bp_at_onset,basis_bp_at_breach,change_window_to_breach_bp
count,83.0,83.00,83.00,83.00,83.00
mean,228.0,-1.24,-0.83,-4.77,-3.52
std,24.1,5.63,8.00,18.10,17.26
min,187.0,-6.77,-12.59,-120.28,-116.26
25%,207.5,-4.74,-4.99,-5.65,-2.16
50%,228.0,-4.07,-3.96,-4.24,-0.57
75%,248.5,0.53,1.45,-1.62,1.35
max,269.0,16.09,33.32,36.88,20.78


## Summary (descriptive)

In [7]:
summary = dict(
    reproduced=REPRO.to_dict('records'), matching_sensitivity=SENS.to_dict('records'),
    unmatched_reasons=UN.groupby(['venue', 'reason']).size().rename('n').reset_index().to_dict('records') if len(UN) else [],
    price_order=PO.to_dict('records'), signal_order=SO.reset_index().rename(columns={'index': 'signal'}).to_dict('records'),
    signal_relative_to_onset=PRE.to_dict('records'), score_cofiring=CO.to_dict('records')[0],
    lead_lag_peak={lab: dict(returns_k=int(g.loc[g.corr_returns.idxmax(), 'lag_k']), abs_returns_k=int(g.loc[g.corr_abs_returns.idxmax(), 'lag_k']))
                   for lab, g in LL.groupby('sample')},
    basis_change_window_to_breach_bp_median=float(BA.change_window_to_breach_bp.median()) if len(BA) else None)
(OUT / 'SAME_EVENT_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print(json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))[:4000])

{
 "reproduced": [
  {
   "venue": "spot",
   "threshold_2022": 0.6698990591069133,
   "events": 83,
   "alarms": 800,
   "caught": 39,
   "pre_onset": 22,
   "post_onset": 17
  },
  {
   "venue": "futures",
   "threshold_2022": 0.6710872482458041,
   "events": 84,
   "alarms": 800,
   "caught": 35,
   "pre_onset": 19,
   "post_onset": 16
  }
 ],
 "matching_sensitivity": [
  {
   "tolerance_min": 15,
   "matched": 82,
   "spot_only": 1,
   "futures_only": 2
  },
  {
   "tolerance_min": 60,
   "matched": 83,
   "spot_only": 0,
   "futures_only": 1
  },
  {
   "tolerance_min": 240,
   "matched": 83,
   "spot_only": 0,
   "futures_only": 1
  }
 ],
 "unmatched_reasons": [
  {
   "venue": "futures",
   "reason": "other venue touched 4% (5-min/other rule failed)",
   "n": 1
  }
 ],
 "price_order": [
  {
   "milestone": "peak",
   "pairs": 83,
   "futures_first": 7,
   "same_minute": 74,
   "spot_first": 2,
   "median_diff_min": 0.0,
   "mean_abs_diff_min": 0.9759036144578314
  },
  {
   "mil